### 导入需要的安装包

In [84]:
import ee
import geemap

### 设置端口信息

In [85]:
# geemap.set_proxy(port=33210)
# ee.Authenticate()
# ee.Initialize(project='ee-scistudy')
ee.Initialize()

### 加载显示一下基础影像，如果出现地图画面就说明成功了

In [86]:
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

### 接下来开始深度学习模型的使用

### 1、导入样本数据

In [87]:
import pandas as pd

# pdData = pd.DataFrame(results, columns=feaProperty)

pdData = pd.read_csv(r"C:\Users\10655\Desktop\test\P3_sample1500.csv")
pdData.head(2)


,landcover,B1,B10,B11,B11_1,B12,B1_1,B2,B2_1,B3,...,VH_var,VV,VV_asm,VV_contrast,VV_corr,VV_ent,VV_idm,VV_var,elevation,slope
0,0,0.163135,310.84033,311.62143,0.33925,0.33835,0.0999,0.157672,0.1301,0.170486,...,69.472666,-9.863521,0.016243,39.891865,0.497342,4.184581,0.187305,40.425783,1877,4
1,0,0.149214,300.60672,300.35425,0.29600,0.32240,0.0886,0.139993,0.0934,0.138076,...,108.948287,-5.940173,0.014261,119.821429,0.394645,4.291166,0.110539,98.709755,1878,10


In [88]:
import copy
pdData2 = pdData.copy()

className = pdData2['landcover'].unique().tolist()
print("className 包含的类别是：\n", className)

className 包含的类别是：
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


In [89]:
# 随机增加一列，用于后续划分训练和测试集
import numpy as np
np.random.seed(42)
pdData2['random'] = np.random.randint(1, 100, pdData2.shape[0])
pdData2.head(2)

,landcover,B1,B10,B11,B11_1,B12,B1_1,B2,B2_1,B3,...,VV,VV_asm,VV_contrast,VV_corr,VV_ent,VV_idm,VV_var,elevation,slope,random
0,0,0.163135,310.84033,311.62143,0.33925,0.33835,0.0999,0.157672,0.1301,0.170486,...,-9.863521,0.016243,39.891865,0.497342,4.184581,0.187305,40.425783,1877,4,52
1,0,0.149214,300.60672,300.35425,0.29600,0.32240,0.0886,0.139993,0.0934,0.138076,...,-5.940173,0.014261,119.821429,0.394645,4.291166,0.110539,98.709755,1878,10,93


In [90]:
train_data = pdData2[pdData2['random'] <= 70]
val_data = pdData2[pdData2['random'] > 30]

In [91]:
featureName = ['B1','B2', 'B3', 'B4', 'B5', 'B6', 
'B7','B8','B10','B9','B11','elevation','slope','B1_1','B2_1','B3_1','B4_1','B5_1'
,'B6_1','B7_1','B8_1','B8A','B9_1','B11_1','B12','PC1','PC2','PC3','VV','VH'
,'VV_asm','VV_idm','VV_ent','VH_asm'
,'VH_idm','VH_ent']
labelName = ['landcover']


def split_train_val_test(pddata,selectedColumn,classColumn):
    data = np.array(pddata[selectedColumn].values.tolist())
    data_label = np.array(pddata[classColumn])
    return data,data_label

trainData,trainData_label =  split_train_val_test(train_data,featureName,labelName)
valData,valData_label =  split_train_val_test(val_data,featureName,labelName)

print("trainData shape:\n",trainData.shape)
print("trainData_label shape:\n",trainData_label.shape)
print("valData shape:\n",valData.shape)
print("valData_label shape:\n",valData_label.shape)

trainData shape:
 (14885, 36)
trainData_label shape:
 (14885, 1)
valData shape:
 (14545, 36)
valData_label shape:
 (14545, 1)


In [92]:
### 把样本集、验证集换成tensor格式

import torch

trainData = torch.tensor(trainData, dtype=torch.double)
trainData_label = torch.tensor(trainData_label, dtype=torch.double)
print(f"trainData shape is {trainData.shape}.\n")
print("trainData_label first element:\n",trainData_label[0])

valData = torch.tensor(valData, dtype=torch.double)
valData_label = torch.tensor(valData_label, dtype=torch.double)
print(f"valData shape is {valData.shape}.\n")




trainData shape is torch.Size([14885, 36]).

trainData_label first element:
 tensor([0.])
valData shape is torch.Size([14545, 36]).



In [93]:
### 将数据集转成torch dataset结构并进行组织

import matplotlib.pyplot as plt
import pandas as pd
import ast
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import ReLU, Sigmoid
torch.set_default_dtype(torch.double)

In [94]:
# 使用TensorDataset构建trainDataset和valDataset
trainDataset = TensorDataset(trainData, trainData_label)
valDataset = TensorDataset(valData, valData_label)

# 定义batch_size
# batch_size = 128

# # 创建 DataLoader
# trainLoader = DataLoader(trainDataset, batch_size=batch_size, shuffle=True)
# valLoader = DataLoader(valDataset, batch_size=batch_size, shuffle=False)  # 在验证集上一般不需要打乱数据

import random
import numpy as np
from torch.utils.data import Sampler, DataLoader

class ClassBalancedBatchSampler(Sampler):
    def __init__(self, labels, num_classes, num_samples_per_class):
        self.num_classes = num_classes
        self.num_samples_per_class = num_samples_per_class
        # 为每个类别收集索引并打乱
        self.class_indices = {
            cls: np.where(labels == cls)[0].tolist()
            for cls in range(num_classes)
        }
        for idxs in self.class_indices.values():
            random.shuffle(idxs)
        # 能组成的完整 batch 数量
        self.num_batches = min(
            len(idxs)//num_samples_per_class
            for idxs in self.class_indices.values()
        )

    def __iter__(self):
        for i in range(self.num_batches):
            batch = []
            for cls in range(self.num_classes):
                start = i * self.num_samples_per_class
                end   = start + self.num_samples_per_class
                batch.extend(self.class_indices[cls][start:end])
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches

# —— 在这里提取标签数组 —— 
labels = trainData_label.squeeze().cpu().numpy().astype(int)

# 构造 sampler：14 个类别，每类 2 个样本
sampler = ClassBalancedBatchSampler(labels, num_classes=14, num_samples_per_class=1)

# 使用 sampler 构造 DataLoader（不再传 batch_size、shuffle）
trainLoader = DataLoader(trainDataset, batch_sampler=sampler)

# 验证集仍可用常规方式
valLoader   = DataLoader(valDataset, batch_size=14, shuffle=False)


In [95]:
class DNNModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DNNModel, self).__init__()
        self.fc1 = nn.Linear(in_channels, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 64)
        self.fc6 = nn.Linear(64, 64)
        self.fc7 = nn.Linear(64, 32)
        self.fc8 = nn.Linear(32, 32)
        self.fc9 = nn.Linear(32, 16)
        self.fc10 = nn.Linear(16, out_channels)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = F.relu(self.fc6(x))
        x = F.relu(self.fc7(x))
        x = F.relu(self.fc8(x))
        x = F.relu(self.fc9(x))
        x = self.fc10(x)  # 输出层不加激活
        return x



# 检查是否有可用的GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {device}")

# 模型实例化
model = DNNModel(36,14).to(device)
# print(model)

import torch.optim as optim

# 损失函数
criterion = nn.CrossEntropyLoss()

# 定义学习率
learnRt = 0.001

# 定义优化器
optimizer = optim.RMSprop(model.parameters(), lr=learnRt)

# optimizer = optim.SGD(model.parameters(), lr=learnRt)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)



device is cuda


In [96]:
import pandas as pd
from sklearn.metrics import f1_score

def train_and_validate(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=10):
    metrics_list = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        all_train_preds = []
        all_train_labels = []

        # === 打印一次 path 中每类样本数 ===
        if epoch == 0:
            # 取第一个 batch
            sample_inputs, sample_labels = next(iter(train_loader))
            unique, counts = np.unique(sample_labels.cpu().numpy(), return_counts=True)
            class_counts = dict(zip(unique, counts))
            print(f"\n=== 当前 Path (Batch size={len(sample_labels)}) 每类样本数: {class_counts} ===\n")

        # === 训练阶段 ===
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long().squeeze(-1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        train_accuracy = 100 * correct_train / total_train
        train_loss = running_loss / len(train_loader)
        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro')

        # === 验证阶段 ===
        model.eval()
        correct_val = 0
        total_val = 0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).long().squeeze(-1)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        val_accuracy = 100 * correct_val / total_val
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')

        scheduler.step()

        print(f'Epoch [{epoch + 1}/{epochs}] - Loss: {train_loss:.4f} | '
              f'Train Acc: {train_accuracy:.2f}% | Val Acc: {val_accuracy:.2f}% | '
              f'Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}')
        
        # 记录指标
        metrics_list.append({
            'Epoch': epoch + 1,
            'Train Loss': train_loss,
            'Train Acc': train_accuracy,
            'Train F1': train_f1,
            'Val Acc': val_accuracy,
            'Val F1': val_f1
        })

    # === 保存到 CSV ===
    df_metrics = pd.DataFrame(metrics_list)
    csv_path = r'C:\Users\10655\Desktop\test\P3_epoch.csv'
    df_metrics.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n训练日志已保存至: {csv_path}")

# 调用训练
train_and_validate(model, trainLoader, valLoader, criterion, optimizer, scheduler, epochs=100)



=== 当前 Path (Batch size=14) 每类样本数: {0.0: 1, 1.0: 1, 2.0: 1, 3.0: 1, 4.0: 1, 5.0: 1, 6.0: 1, 7.0: 1, 8.0: 1, 9.0: 1, 10.0: 1, 11.0: 1, 12.0: 1, 13.0: 1} ===

Epoch [1/100] - Loss: 2.0474 | Train Acc: 23.34% | Val Acc: 21.21% | Train F1: 0.2191 | Val F1: 0.1817
Epoch [2/100] - Loss: 1.7700 | Train Acc: 31.27% | Val Acc: 28.62% | Train F1: 0.3060 | Val F1: 0.2686
Epoch [3/100] - Loss: 1.5388 | Train Acc: 40.10% | Val Acc: 51.41% | Train F1: 0.3980 | Val F1: 0.4913
Epoch [4/100] - Loss: 1.3632 | Train Acc: 46.47% | Val Acc: 53.75% | Train F1: 0.4588 | Val F1: 0.5148
Epoch [5/100] - Loss: 1.2486 | Train Acc: 50.08% | Val Acc: 57.46% | Train F1: 0.4885 | Val F1: 0.5620
Epoch [6/100] - Loss: 1.0952 | Train Acc: 56.52% | Val Acc: 68.83% | Train F1: 0.5624 | Val F1: 0.6693
Epoch [7/100] - Loss: 0.9220 | Train Acc: 64.33% | Val Acc: 73.18% | Train F1: 0.6401 | Val F1: 0.7297
Epoch [8/100] - Loss: 0.7595 | Train Acc: 71.45% | Val Acc: 75.43% | Train F1: 0.7140 | Val F1: 0.7481
Epoch [9/100] - Lo

In [97]:
weights_and_biases = {}
for layer_name, params in model.named_parameters():
    weights_and_biases[layer_name] = params.detach().cpu().numpy().tolist()

In [98]:
# 将字典按顺序转换为列表
weights_and_biases_list = list(weights_and_biases.values())

# 每两个元素组合为一个集合，并保存为列表
param_list = [weights_and_biases_list[i:i+2] for i in range(0, len(weights_and_biases_list), 2)]

print(len(param_list))
# print(param_list[3][1])

10


In [99]:
roi = ee.FeatureCollection('projects/ee-1065531055/assets/paper3_roi2')
Map.addLayer(roi, {'color':'blue'}, 'studyArea', shown=True)
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [100]:
# 2. Sentinel-2 云掩膜函数
def rmS2cloud(image):
    qa = image.select('QA60')
    cloudBitMask  = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask) \
                .toDouble().divide(10000) \
                .copyProperties(image, ['system:time_start','system:time_end'])

# 3. 构造 Sentinel-2 合成影像
LL_clipped = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(roi)
      .filterDate('2021-07-01', '2021-10-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .select(['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','QA60'])
      .map(rmS2cloud)
      .median()
      .clip(roi)
     )

visParam = {
  "min": 0.0,
  "max": 0.3,
  "bands": ['B4', 'B3', 'B2'],
}
bands1 = ["B2","B3","B4","B8","B8A","B11","B12"]
Map.addLayer(LL_clipped, visParam, 'imgS2')
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [101]:
# 4. Sentinel-1 VV/VH 合成
collectionS1_IW = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filterBounds(roi) \
    .filterMetadata('instrumentMode', 'equals', 'IW') \
    .filterDate('2021-07-01', '2021-10-30')


Sentinel_1=collectionS1_IW.median().clip(roi);

In [102]:

# 假设你已经有了 Sentinel-1 合成影像 s1
input_img = Sentinel_1.divide(11/51).add(2040/11).toUint16()

# 计算 GLCM 纹理（3×3 窗口），注意这里是 {'size': 3}
glcm = input_img.glcmTexture(size=3)

# 从glcm中提取各类纹理波段
VV_contrast = glcm.select('VV_contrast')
VV_asm      = glcm.select('VV_asm')
VV_idm      = glcm.select('VV_idm')
VV_ent      = glcm.select('VV_ent')
VV_corr     = glcm.select('VV_corr')
VV_var      = glcm.select('VV_var')

VH_contrast = glcm.select('VH_contrast')
VH_asm      = glcm.select('VH_asm')
VH_idm      = glcm.select('VH_idm')
VH_ent      = glcm.select('VH_ent')
VH_corr     = glcm.select('VH_corr')
VH_var      = glcm.select('VH_var')

In [103]:
# DEM
dataset = ee.Image('USGS/SRTMGL1_003').clip(roi)
terrain = ee.Algorithms.Terrain(dataset)

# 1. 定义去云函数（TOA 影像）
def rmCloud_TOA(image):
    cloud_score = ee.Algorithms.Landsat.simpleCloudScore(image)
    mask = cloud_score.select('cloud').lte(30)
    return image.updateMask(mask)

# 2. 构造 L8 TOA 合成影像
l8SR = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterBounds(roi)
    .filterDate('2020-05-01', '2020-10-30')
    .map(rmCloud_TOA)
    .select(['B1','B2','B3','B4','B5','B6','B7','B8','B9','B10','B11'])
    .median()
    .clip(roi)
)

# 3. 可视化参数
rgbVis = {
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4,
    'bands': ['B4','B3','B2']
}


Map.addLayer(l8SR, visParam, 'l8SR')
Map.centerObject(roi, 8)
Map


Map(center=[41.349157554130265, 96.22375509543464], controls=(WidgetControl(options=['position', 'transparent_…

In [104]:
# 定义一个函数，用于对矢量进行矩形分块
def shpRectBlock(vector, numHorizontal, numVertical):
    if not isinstance(vector, (ee.Geometry, ee.Feature, ee.FeatureCollection)):
        raise ValueError('Input must be a Geometry, Feature, or FeatureCollection.')
    if isinstance(vector, ee.FeatureCollection):
        geom = vector.geometry()
    elif isinstance(vector, ee.Feature):
        geom = vector.geometry()
    else:
        geom = vector
    bounds = geom.bounds()
    coords = bounds.coordinates().get(0)
    xmin = ee.Number(coords.get(0).get(0))
    ymin = ee.Number(coords.get(0).get(1))
    xmax = ee.Number(coords.get(2).get(0))
    ymax = ee.Number(coords.get(2).get(1))
    xStep = xmax.subtract(xmin).divide(numHorizontal)
    yStep = ymax.subtract(ymin).divide(numVertical)
    rects = (
        ee.List.sequence(0, numHorizontal - 1)
          .map(lambda i: ee.List.sequence(0, numVertical - 1)
            .map(lambda j: ee.Feature(
                ee.Geometry.Rectangle([
                    xmin.add(xStep.multiply(i)),
                    ymin.add(yStep.multiply(j)),
                    xmin.add(xStep.multiply(i)).add(xStep),
                    ymin.add(yStep.multiply(j)).add(yStep)
                ]).intersection(geom, 1)
            ))
          )
          .flatten()
    )
    return ee.FeatureCollection(rects).filterBounds(geom)

# 定义 PCA 函数
def pca(img):
    bandNames = bands1
    scale = 200
    region = roi

    # 1. 计算均值并中心化
    meanDict = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e13
    )
    means = ee.Image.constant(meanDict.values(bandNames))
    centered = img.subtract(means)

    # 辅助：生成 PC 波段名（PC1…PC7）
    def getBandNames(prefix):
        return ee.List.sequence(1, len(bandNames)) \
                 .map(lambda n: ee.String(prefix).cat(
                     ee.Number(n).format('%.0f')
                 ))

    # 2. 计算协方差并做特征分解
    arrays = centered.toArray()
    weights = ee.Array([2] + [1] * (len(bandNames) - 1))
    arrays = arrays.multiply(weights)

    covar = arrays.reduceRegion(
        reducer=ee.Reducer.centeredCovariance(),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e13
    )
    covarArr = ee.Array(covar.get('array'))
    eigens = covarArr.eigen()
    eigenValues  = eigens.slice(1, 0, 1)
    eigenVectors = eigens.slice(1, 1)

    # 3. 投影到主成分空间并标准化
    arrImg = arrays.toArray(1)
    pcArr  = ee.Image(eigenVectors).matrixMultiply(arrImg)

    sdImg = ee.Image(eigenValues.sqrt()) \
        .arrayProject([0]) \
        .arrayFlatten([getBandNames('SD')])

    pcImage = (
        pcArr
        .arrayProject([0])
        .arrayFlatten([getBandNames('PC')])
        .divide(sdImg)
        .set('eigenValues', eigenValues.toList().flatten())
        .set('eigenVectors', eigenVectors)
    )
    return pcImage

# 应用 PCA 并可视化
pcImage = pca(LL_clipped.select(bands1))

Map = geemap.Map(center=[0, 0], zoom=6)
Map.addLayer(
    pcImage,
    {'bands': ['PC3', 'PC2', 'PC1'], 'min': -2, 'max': 2},
    'Landsat 8 - PCA'
)
Map.addLayerControl()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [105]:
LL_clipped2 = ee.Image.cat([
    l8SR,
    LL_clipped,
    terrain,
    pcImage,
    Sentinel_1
]).toFloat().clip(roi)

# 2. 在主体影像后面添加 VV/VH 纹理波段
LL_clipped1 = LL_clipped2.addBands([
    VV_contrast, VV_asm, VV_idm, VV_ent, VV_corr, VV_var,
    VH_contrast, VH_asm, VH_idm, VH_ent, VH_corr, VH_var
])

In [106]:
def dnnMultiply(inputImg,nn_weights,nn_bias,activate_boolean=False):
    
  # 把nn_weights和nn_bias都转成imageArray的形式

  nn_weights = ee.Image(ee.Array(nn_weights))#.toArray().toArray(1)
  nn_bias = ee.Image(ee.Array(nn_bias)).toArray(1)
  imgDNN = nn_weights.matrixMultiply(inputImg).add(nn_bias)

  if activate_boolean:
      return imgDNN.multiply(imgDNN.gt(0))
  else:
      return imgDNN
  # return imgDNN

In [107]:

inputImg = LL_clipped1.select(featureName).toArray().toArray(1)
imgPredict = ee.Image(1)
layer = 1

for layer, param in enumerate(param_list):
    # print(i)
    weightParam, biasParam = param
    # print(len(biasParam))
    print(f"正在运行第{layer+1}层的预测,总共有{len(param_list)}层需要预测")
    # print(len(weightParam))
    # print(len(biasParam))
    if layer == 0:
        imgPredict = dnnMultiply(inputImg,weightParam,biasParam,activate_boolean=True)

    elif layer == len(param_list)-1:
        imgPredict = dnnMultiply(imgPredict,weightParam,biasParam,activate_boolean=False)

    else:
        imgPredict = dnnMultiply(imgPredict,weightParam,biasParam,activate_boolean=True)



正在运行第1层的预测,总共有10层需要预测
正在运行第2层的预测,总共有10层需要预测
正在运行第3层的预测,总共有10层需要预测
正在运行第4层的预测,总共有10层需要预测
正在运行第5层的预测,总共有10层需要预测
正在运行第6层的预测,总共有10层需要预测
正在运行第7层的预测,总共有10层需要预测
正在运行第8层的预测,总共有10层需要预测
正在运行第9层的预测,总共有10层需要预测
正在运行第10层的预测,总共有10层需要预测


In [108]:
# 定义 Softmax 函数
def softmax(arr):
  expArr = arr.exp()
  sumExp = expArr.reduce(ee.Reducer.sum())
  return expArr.divide(sumExp)

expArr = imgPredict.exp()
length = imgPredict.arrayLength(0)

sumExp = expArr.arrayReduce(ee.Reducer.sum(),[0]).arrayRepeat(0,length)

classLabel = expArr.divide(sumExp).arrayProject([0]).arrayArgmax().arrayFlatten([['class']])


In [109]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import torch

# —— 1. 先收集所有验证集的真实标签和预测标签 —— 
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in valLoader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# —— 2. 计算指标 —— 
oa = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
report = classification_report(y_true, y_pred, digits=4)
cm = confusion_matrix(y_true, y_pred)

# —— 3. 将混淆矩阵做成 DataFrame 方便查看 —— 
# 假设您的类别名列表如下（0–13 共 14 类），可替换为实际名称
class_names = [str(i) for i in range(14)]
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

# —— 4. 输出 —— 
print(f"Overall Accuracy (OA): {oa:.4f}")
print(f"Weighted F1 Score: {f1:.4f}\n")
print("Classification Report:\n", report)
print("\nConfusion Matrix:")
display(cm_df)


Overall Accuracy (OA): 0.9278
Weighted F1 Score: 0.9278

Classification Report:
               precision    recall  f1-score   support

         0.0     0.9098    0.8361    0.8714      1025
         1.0     0.9960    0.9970    0.9965      1011
         2.0     0.9962    0.9933    0.9947      1043
         3.0     0.9622    0.9559    0.9590      1065
         4.0     0.9904    0.9952    0.9928      1039
         5.0     0.9754    0.9745    0.9749      1057
         6.0     0.8617    0.8902    0.8757      1029
         7.0     0.9716    0.9671    0.9694      1063
         8.0     0.9546    0.9537    0.9542      1037
         9.0     0.8314    0.8092    0.8201      1048
        10.0     0.8049    0.8478    0.8258      1051
        11.0     0.8335    0.8343    0.8339      1020
        12.0     0.9384    0.9575    0.9479      1035
        13.0     0.9671    0.9765    0.9718      1022

    accuracy                         0.9278     14545
   macro avg     0.9281    0.9277    0.9277     14545

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,857,1,0,0,0,0,2,0,0,22,45,97,1,0
1,2,1008,0,0,0,0,0,0,1,0,0,0,0,0
2,0,0,1036,0,0,0,0,0,1,0,0,0,1,5
3,0,0,0,1018,5,0,0,0,42,0,0,0,0,0
4,0,0,0,2,1034,0,0,0,3,0,0,0,0,0
5,0,0,0,0,0,1030,0,0,0,0,0,0,0,27
6,4,0,0,0,0,2,916,0,0,80,26,0,1,0
7,0,0,0,0,0,0,0,1028,0,0,0,3,32,0
8,0,3,0,38,5,0,0,0,989,0,0,0,0,2
9,17,0,1,0,0,1,94,0,0,848,53,6,28,0


In [110]:
# Define a palette for the IGBP classification.
visPalette = [
  '#BFBFBF','#F9F3C0','#3C8A5B','#A4CA73',
  '#007AFF','#C14646','#FFAA00','#00CC99',
  '#9900CC','#FF33CC','#33CCFF','#666600',
  '#CC6666','#0066CC'
]
visParam2={'palette':visPalette, "min":0,"max":13}
Map.addLayer(classLabel,visParam2,'imgLULC')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [111]:
# 批处理导出到 Google Drive
task = ee.batch.Export.image.toDrive(
    image=classLabel,
    description='export_classLabel1',
    folder='gee_exports1',          # Drive 上的文件夹名
    fileNamePrefix='classLabel1',   # 导出文件前缀
    scale=10,
    region=roi.geometry(),
    maxPixels=1e13
)
task.start()
print("已启动 Drive 导出任务，稍后在 Google Drive 的gee_exports文件夹中查看。")

已启动 Drive 导出任务，稍后在 Google Drive 的gee_exports文件夹中查看。
